# P3: Acquisition & Integration

**DSA 405 · Fall 2026 · Section 002 · Avery Atkinson (alatkin3)**

**Project question:** Do restaurants in lower-income NYC zip codes get more critical health-code violations than restaurants in higher-income zip codes?

**Sources**
1. NYC DOHMH Restaurant Inspection Results, NYC Open Data (Socrata API, dataset `43nn-pn8j`)
2. U.S. Census Bureau, ACS 5-year Subject Table S1901, median household income (`S1901_C01_012E`) by ZIP Code Tabulation Area (ZCTA)

This notebook executes the A8 acquisition plan: it collects both sources with rate-limited, cached, fail-loud code, joins them on zip code, and verifies the join.

*This product uses the Census Bureau Data API but is not endorsed or certified by the Census Bureau.*

**To run:** add your Census key as a Colab Secret named `CENSUS_KEY` (key icon in the left sidebar, Notebook access on), then Runtime > Restart and run all. The first run makes about 10 requests and takes about a minute. Re-runs read everything from `data/raw/` and make no new requests, except the deliberate failure demo in Section 1.6.

In [1]:
# Setup
import os, io, json, time
import numpy as np, pandas as pd, requests

RAW = "data/raw"; CLEAN = "data/clean"
os.makedirs(RAW, exist_ok=True); os.makedirs(CLEAN, exist_ok=True)

# Honest User-Agent: says who is asking and why
HEADERS = {"User-Agent": "DSA405-student-coursework (NC State University class project; non-commercial)"}
PAUSE = 2.0   # seconds between requests (A8 plan: at most 1 request every 2 seconds)

pd.set_option("display.width", 160)
print("pandas", pd.__version__)

pandas 2.2.3


---
# 1. Acquisition code

## 1.1 The fetch functions

Every request goes through `polite_get`, and every saved file goes through `cached_fetch`.

| Requirement | Where it is handled |
|---|---|
| Rate limited | `polite_get` waits until `PAUSE` seconds have passed since the last request, and waits longer if the server's rate-limit headers say so (Section 1.3) |
| Honest User-Agent | `HEADERS`, sent with every request |
| Cached | `cached_fetch` saves the raw response to `data/raw/` and reads from there on re-runs. A file is saved only after it passes its checks, so a bad response is never cached. |
| Pagination | Section 1.4, using Socrata's `$limit` / `$offset` |
| Timeouts | `timeout=60` on every request; a timeout or dropped connection is retried, then raises |
| Non-200 responses | 429 and 5xx are retried with a longer wait each time; any other non-200 raises `FetchError` at once |
| Selectors that match nothing | For an API, the equivalent is a response that is empty or is missing the columns the code expects. `check_columns` and the per-source checks raise `FetchError` instead of continuing. |
| Fails loudly | Every problem above raises an error with a message. Section 1.6 demonstrates it. |

In [2]:
class FetchError(RuntimeError):
    """Raised whenever a response could silently produce wrong data."""

_last_request = [0.0]
_extra_wait = [0.0]   # set from rate-limit headers, see rate_limit_from_headers()

def polite_get(url, params=None, tries=4):
    """One GET request: rate limited, honest User-Agent, timeout, retries on 429/5xx, raises on anything else."""
    for attempt in range(1, tries + 1):
        wait = PAUSE + _extra_wait[0] - (time.time() - _last_request[0])
        if wait > 0:
            time.sleep(wait)
        _last_request[0] = time.time()
        try:
            r = requests.get(url, params=params, headers=HEADERS, timeout=60)
        except (requests.Timeout, requests.ConnectionError) as e:
            print(f"  attempt {attempt}: {type(e).__name__}; waiting {5*attempt}s and retrying")
            time.sleep(5 * attempt)
            continue
        rate_limit_from_headers(r.headers, quiet=True)
        if r.status_code == 429 or r.status_code >= 500:
            print(f"  attempt {attempt}: HTTP {r.status_code}; waiting {5*attempt}s and retrying")
            time.sleep(5 * attempt)
            continue
        if r.status_code != 200:
            raise FetchError(f"HTTP {r.status_code} from {url} -> {r.text[:200]!r}")
        return r
    raise FetchError(f"Gave up after {tries} attempts: {url}")

def cached_fetch(name, url, params=None, check=None):
    """Return the raw text for one resource. Reads data/raw/<name> if it exists (no request).
    Otherwise fetches it, runs check(response) and saves it only if the check passes."""
    path = os.path.join(RAW, name)
    if os.path.exists(path):
        print(f"{name}: read from cache, no request")
        with open(path, encoding="utf-8") as f:
            return f.read()
    r = polite_get(url, params)
    if check is not None:
        check(r)                      # raises FetchError on a bad response; nothing is saved
    with open(path, "w", encoding="utf-8") as f:
        f.write(r.text)
    with open(path + ".headers.json", "w") as f:   # keep the response headers for Section 1.3
        json.dump(dict(r.headers), f, indent=1)
    print(f"{name}: HTTP {r.status_code}, {len(r.content):,} bytes, saved to {path}")
    return r.text

def check_columns(df, needed, label):
    """The API version of 'a selector that matched nothing': empty data or missing columns."""
    if df is None or len(df) == 0:
        raise FetchError(f"{label}: response contained no rows")
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise FetchError(f"{label}: expected columns not found: {missing}. Columns present: {list(df.columns)[:10]}")

## 1.2 Census API key

The key is stored as a Colab Secret, so it never appears in the notebook. `.split()[0]` keeps only the first copy of the key (in P2 the secret had been pasted twice, making it 81 characters). The key is sent in `params` and never printed.

In [3]:
try:
    from google.colab import userdata
    CENSUS_KEY = userdata.get("CENSUS_KEY").split()[0]
except ImportError:                       # running outside Colab
    import getpass
    CENSUS_KEY = getpass.getpass("Census API key: ").split()[0]

# fail loudly: a wrong-length key gets an 'Invalid Key' web page back, not data
if len(CENSUS_KEY) != 40:
    raise ValueError(f"Census key should be 40 characters, got {len(CENSUS_KEY)}")
print("Census key loaded (40 characters)")

Census key loaded (40 characters)


## 1.3 Rate limit, read from the response headers

The documentation can be out of date; the headers state the limit that applies right now. `rate_limit_from_headers` looks for the standard headers (`X-RateLimit-Limit`, `X-RateLimit-Remaining`, `X-RateLimit-Reset`, `Retry-After`). `polite_get` calls it after **every** response. If `Retry-After` is present, or fewer than 10 requests remain, it adds an extra wait and prints a warning. If the server sends no rate-limit headers at all, the code says so and keeps the A8 pause of 2 seconds.

This first request asks NYC Open Data for the total row count (one small request). The count is also used in Section 1.4 to check that pagination collected every row.

In [4]:
def rate_limit_from_headers(headers, quiet=False):
    """Find rate-limit headers and adjust the wait. Returns the headers it found."""
    h = {k.lower(): v for k, v in dict(headers).items()}
    found = {k: v for k, v in h.items() if "ratelimit" in k or "rate-limit" in k or k == "retry-after"}
    _extra_wait[0] = 0.0
    if "retry-after" in h:
        try:
            _extra_wait[0] = float(h["retry-after"])
            print(f"  WARNING: server sent Retry-After: {h['retry-after']} s; waiting that long before the next request")
        except ValueError:
            pass
    remaining = h.get("x-ratelimit-remaining")
    if remaining is not None and remaining.isdigit() and int(remaining) < 10:
        _extra_wait[0] = max(_extra_wait[0], 10.0)
        print(f"  WARNING: only {remaining} requests left in this window; slowing down")
    if not quiet:
        if found:
            print("Rate-limit headers sent by the server:")
            for k, v in found.items():
                print(f"  {k}: {v}")
        else:
            print("The server sent NO rate-limit headers, so there is no stated limit to read.")
            print(f"Keeping the A8 plan's pause of {PAUSE} s between requests (stricter than robots.txt Crawl-delay: 1).")
    return found

NYC = "https://data.cityofnewyork.us/resource/43nn-pn8j"

def check_json(r):
    if "json" not in r.headers.get("Content-Type", ""):
        raise FetchError(f"Expected JSON, got {r.headers.get('Content-Type')}: {r.text[:200]!r}")

count_txt = cached_fetch("nyc_rowcount.json", NYC + ".json", {"$select": "count(*)"}, check=check_json)
n_total = int(list(json.loads(count_txt)[0].values())[0])
print(f"NYC Open Data says the dataset has {n_total:,} rows\n")

# read the rate limit from the headers that came back with that response
with open(os.path.join(RAW, "nyc_rowcount.json.headers.json")) as f:
    nyc_headers = json.load(f)
print("NYC Open Data:")
nyc_rl = rate_limit_from_headers(nyc_headers)

nyc_rowcount.json: read from cache, no request
NYC Open Data says the dataset has 294,959 rows

NYC Open Data:
The server sent NO rate-limit headers, so there is no stated limit to read.
Keeping the A8 plan's pause of 2.0 s between requests (stricter than robots.txt Crawl-delay: 1).


## 1.4 Source 1: NYC restaurant inspections (paginated)

Socrata returns at most `$limit` rows per request, so the code walks through the dataset with `$offset` in pages of 50,000 rows. `$order=:id` keeps the order fixed so no row is skipped or repeated between pages. Each page is cached as its own file. The loop stops at the first page with fewer than 50,000 rows, and then checks that the pages add up to the row count from Section 1.3.

In [5]:
LIMIT = 50_000
NEED_NYC = ["camis", "dba", "boro", "zipcode", "inspection_date", "critical_flag", "score", "violation_code"]

def check_csv(r):
    if "csv" not in r.headers.get("Content-Type", ""):
        raise FetchError(f"Expected CSV, got {r.headers.get('Content-Type')}: {r.text[:200]!r}")

pages, offset, i = [], 0, 0
while True:
    txt = cached_fetch(f"nyc_inspections_page{i:02d}.csv", NYC + ".csv",
                       {"$limit": LIMIT, "$offset": offset, "$order": ":id"}, check=check_csv)
    page = pd.read_csv(io.StringIO(txt), dtype=str)    # dtype=str keeps values exactly as published
    if i == 0:
        check_columns(page, NEED_NYC, "NYC page 0")     # fail loudly if the fields were renamed
    pages.append(page)
    if len(page) < LIMIT:
        break
    offset += LIMIT; i += 1
    if i > 30:
        raise FetchError("More than 30 pages: pagination is not ending, stopping instead of looping forever")

nyc_raw = pd.concat(pages, ignore_index=True)
print(f"\n{len(pages)} pages, {len(nyc_raw):,} rows, {nyc_raw.shape[1]} columns")

# fail loudly: pagination must collect exactly the number of rows the server reports
if len(nyc_raw) != n_total:
    raise FetchError(f"Pagination collected {len(nyc_raw):,} rows but the server reports {n_total:,}. "
                     "The dataset may have updated mid-run: delete data/raw/nyc_* and re-run.")
print("Row count matches the server's count")

nyc_inspections_page00.csv: read from cache, no request
nyc_inspections_page01.csv: read from cache, no request
nyc_inspections_page02.csv: HTTP 200, 28,647,236 bytes, saved to data/raw/nyc_inspections_page02.csv
nyc_inspections_page03.csv: HTTP 200, 28,633,630 bytes, saved to data/raw/nyc_inspections_page03.csv
nyc_inspections_page04.csv: HTTP 200, 28,661,395 bytes, saved to data/raw/nyc_inspections_page04.csv
nyc_inspections_page05.csv: HTTP 200, 25,720,412 bytes, saved to data/raw/nyc_inspections_page05.csv

6 pages, 294,959 rows, 27 columns
Row count matches the server's count


## 1.5 Source 2: Census median household income by ZCTA

One request for every ZCTA in the U.S. The Census API can return HTTP 200 with an "Invalid Key" web page instead of data (this happened in P1), so `check_census` refuses anything that is not JSON with the expected header row.

In [6]:
CENSUS = "https://api.census.gov/data/2023/acs/acs5/subject"
INCOME_VAR = "S1901_C01_012E"
GEO = "zip code tabulation area"

def check_census(r):
    if "json" not in r.headers.get("Content-Type", ""):
        raise FetchError(f"Census sent {r.headers.get('Content-Type')} instead of JSON "
                         f"(often an Invalid Key page): {r.text[:150]!r}")
    header = r.json()[0]
    if INCOME_VAR not in header or GEO not in header:
        raise FetchError(f"Census header row is missing expected columns: {header}")

cen_txt = cached_fetch("census_s1901_zcta_2023.json", CENSUS,
                       {"get": f"NAME,{INCOME_VAR}", "for": f"{GEO}:*", "key": CENSUS_KEY},
                       check=check_census)
cen_json = json.loads(cen_txt)
census_raw = pd.DataFrame(cen_json[1:], columns=cen_json[0])
check_columns(census_raw, ["NAME", INCOME_VAR, GEO], "Census")
print(f"Census rows: {len(census_raw):,}\n")

with open(os.path.join(RAW, "census_s1901_zcta_2023.json.headers.json")) as f:
    census_headers = json.load(f)
print("Census API:")
census_rl = rate_limit_from_headers(census_headers)

census_s1901_zcta_2023.json: HTTP 200, 1,135,574 bytes, saved to data/raw/census_s1901_zcta_2023.json
Census rows: 33,772

Census API:
The server sent NO rate-limit headers, so there is no stated limit to read.
Keeping the A8 plan's pause of 2.0 s between requests (stricter than robots.txt Crawl-delay: 1).


## 1.6 Demonstrated failure: the code fails loudly

**Failure 1 (a live request).** Ask the Census API for a variable that does not exist (`S1901_C01_999E`). The server answers HTTP 400. If the code ignored that, the result would be an empty or garbage table with no message. Instead, `polite_get` raises `FetchError`, and nothing is written to the cache.

**Failure 2 (no request).** Pretend NYC renamed `zipcode` to `zip_code`. This is the API version of a selector that matches nothing: the join key would silently become all-missing. `check_columns` stops with an error instead.

Each failure is wrapped in `try/except` only so the notebook can keep running after the demonstration; the printed message is the error the code raised.

In [7]:
# Failure 1: a bad variable name -> HTTP 400 -> FetchError
try:
    cached_fetch("DEMO_bad_variable.json", CENSUS,
                 {"get": "NAME,S1901_C01_999E", "for": f"{GEO}:10001", "key": CENSUS_KEY},
                 check=check_census)
    print("NOT CAUGHT: the bad request was accepted")          # should never print
except FetchError as e:
    print("CAUGHT, failed loudly:\n ", e)
print("Bad response saved to the cache?", os.path.exists(os.path.join(RAW, "DEMO_bad_variable.json")))

# Failure 2: an expected column is missing -> FetchError
renamed = nyc_raw.head(100).rename(columns={"zipcode": "zip_code"})
try:
    check_columns(renamed, NEED_NYC, "NYC (simulated renamed field)")
    print("NOT CAUGHT")
except FetchError as e:
    print("\nCAUGHT, failed loudly:\n ", e)

CAUGHT, failed loudly:
  HTTP 400 from https://api.census.gov/data/2023/acs/acs5/subject -> "error: unknown variable 'S1901_C01_999E'"
Bad response saved to the cache? False

CAUGHT, failed loudly:
  NYC (simulated renamed field): expected columns not found: ['zipcode']. Columns present: ['camis', 'dba', 'boro', 'building', 'street', 'zip_code', 'phone', 'cuisine_description', 'inspection_date', 'action']


---
# 2. Ethics and legality in practice

## 2.1 Plan vs. what the code does

| A8 plan | P3 code | Same? |
|---|---|---|
| NYC Open Data Socrata API `43nn-pn8j`, `requests` | Same dataset and API, `requests` | Yes |
| About 296,514 rows, 50,000 per request, about 6 requests, run once and cached | 50,000 per page with `$offset`; the number of pages and rows printed in 1.4; cached in `data/raw/` | Yes |
| At most 1 request every 2 seconds | `PAUSE = 2.0` on every request, plus extra waits if rate-limit headers ask for them | Yes |
| Honest User-Agent | `DSA405-student-coursework (NC State University class project; non-commercial)` | Yes |
| Drop phone and street address | Only `camis, dba, boro, zipcode, cuisine_description` are kept in the joined table | Yes |
| Census ACS 5-year S1901, `S1901_C01_012E`, 2023, by ZCTA, free key, no login | Same | Yes |
| Census: 1 request listing only the NYC ZCTAs | 1 request for **all** ZCTAs, filtered by the join | **Drift** |
| Responses cached in `cache/` | Cached in `data/raw/` | **Drift** |
| NYC data as JSON | NYC data as CSV from the same API | **Drift** |
| (not in plan) | One extra small request for the row count, and the deliberate failure request in 1.6 | **Drift** |

## 2.2 Drift, and why

1. **Census: all ZCTAs instead of only NYC's.** Listing NYC zip codes in the request would fail the whole request if any one of them is not a Census ZCTA, and some NYC zip codes are not ZCTAs (Section 5). Asking for all ZCTAs is still one request, and it is the same public table. The extra rows are dropped by the left join.
2. **`data/raw/` instead of `cache/`.** The P3 instructions require raw responses in `data/raw/`. The behavior is unchanged: never fetch the same resource twice.
3. **CSV instead of JSON for NYC.** Same endpoint and same rows, but CSV is much smaller for about 300,000 rows, and `dtype=str` keeps every value exactly as published.
4. **Two extra requests.** The row-count request lets the code prove pagination is complete. The failure demo is required by the rubric. Both are single small requests, at the same 2-second pace.
5. **Rate limit.** A8 took the limit from robots.txt and Socrata's documentation. P3 reads it from the response headers. What those headers said is printed in 1.3. If no limit was stated, the A8 pace stays in place.

## 2.3 Guardrails, re-confirmed

- [x] **No login.** Both are public APIs. The Census key is free, needs no account or password, and was requested with an email only.
- [x] **No paywall.**
- [x] **No personal data.** Records describe restaurants and ZIP-level aggregate income. Phone numbers and street addresses are not kept.
- [x] **No circumvention.** No rate limit, block or bot check is bypassed. The code identifies itself honestly, slows down when told to, and stops on errors instead of hammering the server.
- [x] robots.txt allowed both exact paths (A8 Task 1.3: NYC `allowed: True`, `Crawl-delay: 1`; Census `allowed: True`).
- [x] Census attribution notice is at the top of this notebook.

---
# 3. Clean only what the join needs

These steps repeat the P2 cleaning decisions that affect the join key and the income values. The raw files in `data/raw/` are never modified.

In [8]:
df = nyc_raw.copy()
df["inspection_date"] = pd.to_datetime(df["inspection_date"], errors="coerce")

# P2 decision: 1900-01-01 is a placeholder for 'never inspected', not an inspection
placeholder = df["inspection_date"] == pd.Timestamp("1900-01-01")
print(f"placeholder (never-inspected) rows removed: {int(placeholder.sum()):,}")
df = df[~placeholder]

# zip codes: text, trimmed; must be exactly 5 digits or missing (fail loudly otherwise)
df["zipcode"] = df["zipcode"].str.strip()
bad_zip = df["zipcode"].notna() & ~df["zipcode"].str.fullmatch(r"\d{5}", na=False)
if bad_zip.any():
    raise ValueError(f"{int(bad_zip.sum())} zip codes are not 5 digits, e.g. {df.loc[bad_zip,'zipcode'].unique()[:5]}")

# LEFT table: one row per restaurant, using its most recent row (as in P2's tidy restaurants table)
restaurants = (df.sort_values("inspection_date")
                 .drop_duplicates("camis", keep="last")
                 [["camis", "dba", "boro", "zipcode", "cuisine_description"]]
                 .reset_index(drop=True))
print(f"restaurants: {len(restaurants):,} rows, {restaurants.camis.nunique():,} distinct camis, "
      f"{int(restaurants.zipcode.isna().sum()):,} with no zip code")

# RIGHT table: one row per ZCTA
income = census_raw.rename(columns={GEO: "zcta", INCOME_VAR: "median_household_income"})[
    ["zcta", "median_household_income"]].copy()
num = pd.to_numeric(income["median_household_income"], errors="coerce")
if num.isna().any():
    raise ValueError("Some income values are not numbers")
income["median_household_income"] = num
coded = income["median_household_income"] < 0          # -666666666 = 'not available'
income.loc[coded, "median_household_income"] = np.nan
income["income_topcoded"] = income["median_household_income"] == 250001
print(f"income: {len(income):,} ZCTAs, {income.zcta.nunique():,} distinct, "
      f"{int(coded.sum()):,} 'not available' codes set to missing, {int(income.income_topcoded.sum())} top-coded")

restaurants.to_csv(f"{CLEAN}/restaurants_for_join.csv", index=False)
income.to_csv(f"{CLEAN}/census_income_2023_clean.csv", index=False)

placeholder (never-inspected) rows removed: 3,992
restaurants: 27,463 rows, 27,463 distinct camis, 358 with no zip code
income: 33,772 ZCTAs, 33,772 distinct, 3,154 'not available' codes set to missing, 86 top-coded


---
# 4. Join design

## 4.1 Written before the merge

| | |
|---|---|
| **Join key(s)** | `restaurants.zipcode` = `income.zcta` (both 5-character text) |
| **Grain of the left table** | one row per restaurant (`camis`) |
| **Grain of the right table** | one row per ZIP Code Tabulation Area (`zcta`) |
| **Relationship** | many-to-one: many restaurants share a zip code; each ZCTA appears once in the Census table |
| **Expected row count after the join** | exactly the number of restaurants: **27,463** in my run on Nov 5, 2026 (printed below as `n_expected`; NYC updates the data daily, so a later run can differ slightly) |
| **Join type and why** | **left**. Every restaurant must stay in the data, including those whose zip has no Census match, so dropping them is a visible, counted decision for later, not a silent side effect of the join |
| **What happens to unmatched rows** | Restaurants with no matching ZCTA, or no zip code, are kept with income = NA and `_merge = left_only`, and they are counted below. ZCTAs with no restaurant (most of the U.S.) are discarded, as a left join should. |

## 4.2 Alternatives ruled out

- **Inner join:** would silently drop every restaurant whose zip is missing or is not a ZCTA (Week 6's "wreck"). The number dropped would never be reported.
- **Outer join:** would add about 33,000 ZCTAs with no restaurant, rows that do not belong in a table of restaurants.
- **Right join:** the grain would become ZCTAs, which is the wrong unit for this question.
- **Joining the inspections or violations table directly:** also many-to-one and valid, but the restaurant table is the cleanest place to prove the key works. Inspections can be joined to this result by `camis` later.
- **Key choice:** `community_board` (the A8 fallback) is not needed: 26,959 of the 27,105 restaurants with a zip code (99.5%) matched a ZCTA (Section 6).

## 4.3 Which failure this guards against

**Both.**
- **Fan-out (adding rows):** `validate="m:1"` makes pandas raise an error if any ZCTA appears more than once on the right. The row-count check in `verify()` catches it a second way.
- **Wreck (removing rows):** the left join cannot drop restaurants, and `verify()` checks that every `camis` survives and that the match rate did not collapse (which is what happens if, for example, one key is stored as a number and the other as text).

In [9]:
n_expected = len(restaurants)
print("n_expected =", n_expected)

joined = restaurants.merge(income, left_on="zipcode", right_on="zcta",
                           how="left", validate="m:1", indicator=True)

print("rows after join:", len(joined))
print(joined["_merge"].value_counts().to_string())

n_expected = 27463
rows after join: 27463
_merge
both          26959
left_only       504
right_only        0


---
# 5. Verification suite

`verify()` returns a list of failures, so it can be re-run after any change. Each check states the assumption it protects. The suite is then run on two deliberately broken joins to prove it catches both a fan-out and a wreck.

In [10]:
def verify(df, n_expected, left, right, min_match=0.95):
    """Each check states an assumption about the data. If an assumption stops being true, this function reports it."""
    fails = []

    # 1. Row count. Protects against: a fan-out join silently multiplying restaurants.
    if len(df) != n_expected:
        fails.append(f"row count {len(df):,} != expected {n_expected:,}")

    # 2. Key uniqueness. Protects against: the same restaurant appearing twice after the join.
    dup = int(df["camis"].duplicated().sum())
    if dup:
        fails.append(f"{dup} camis values appear more than once")

    # 3. Cardinality. Protects against: the right table having repeated ZCTAs, which breaks many-to-one.
    dup_z = int(right["zcta"].duplicated().sum())
    if dup_z:
        fails.append(f"{dup_z} ZCTAs appear more than once in the income table")

    # 4. Value range. Protects against: Census codes such as -666666666 or impossible incomes leaking in.
    inc = df["median_household_income"].dropna()
    out = int(((inc <= 0) | (inc > 250001)).sum())
    if out:
        fails.append(f"{out} incomes outside (0, 250,001]")

    # 5. Nothing lost. Protects against: a wreck, where restaurants vanish in the join.
    lost = set(left["camis"]) - set(df["camis"])
    if lost:
        fails.append(f"{len(lost):,} restaurants from the left table are missing after the join")

    # 6. Referential integrity / match rate. Protects against: a silent key mismatch
    #    (e.g., zip stored as a number on one side and text on the other) that leaves almost everything unmatched.
    has_zip = df["zipcode"].notna()
    rate = df.loc[has_zip, "zcta"].notna().mean() if has_zip.any() else 0
    if rate < min_match:
        fails.append(f"only {rate:.1%} of restaurants with a zip code matched a ZCTA (minimum {min_match:.0%})")

    # 7. Key type. Protects against: zip codes losing leading zeros or becoming floats ('10001.0').
    for name, s in [("left zipcode", left["zipcode"].dropna()), ("right zcta", right["zcta"])]:
        if not s.astype(str).str.fullmatch(r"\d{5}").all():
            fails.append(f"{name} has values that are not 5-digit text")

    return fails

fails = verify(joined, n_expected, restaurants, income)
print("Failures:", fails if fails else "none, all 7 checks pass")
assert not fails, fails      # stop the notebook if the real join breaks any assumption

Failures: none, all 7 checks pass


### Proof the suite catches real failures

- **Broken join A (fan-out):** the income table with a few ZCTAs duplicated, merged *without* `validate=`. Restaurants in those zips are repeated.
- **Broken join B (wreck):** an inner join, which silently drops restaurants with no match.
- **And `validate=` itself:** the same duplicated table with `validate="m:1"` raises `MergeError` before any wrong table exists.

In [11]:
from pandas.errors import MergeError

# A: fan-out (duplicate 3 ZCTAs that have restaurants, no validate=)
busy = joined["zcta"].value_counts().index[:3]
income_dup = pd.concat([income, income[income.zcta.isin(busy)]], ignore_index=True)
broken_a = restaurants.merge(income_dup, left_on="zipcode", right_on="zcta", how="left")
print("A (fan-out):")
for f in verify(broken_a, n_expected, restaurants, income_dup):
    print("  CAUGHT:", f)

# B: wreck (inner join)
broken_b = restaurants.merge(income, left_on="zipcode", right_on="zcta", how="inner")
print("\nB (wreck, inner join):")
for f in verify(broken_b, n_expected, restaurants, income):
    print("  CAUGHT:", f)

# validate= catches A before it happens
try:
    restaurants.merge(income_dup, left_on="zipcode", right_on="zcta", how="left", validate="m:1")
except MergeError as e:
    print("\nvalidate='m:1' CAUGHT:", e)

A (fan-out):
  CAUGHT: row count 29,399 != expected 27,463
  CAUGHT: 1936 camis values appear more than once
  CAUGHT: 3 ZCTAs appear more than once in the income table

B (wreck, inner join):
  CAUGHT: row count 26,959 != expected 27,463
  CAUGHT: 504 restaurants from the left table are missing after the join

validate='m:1' CAUGHT: Merge keys are not unique in right dataset; not a many-to-one merge


---
# 6. What the join shows, and one surprise

In [12]:
has_zip = joined["zipcode"].notna()
no_match = joined[has_zip & joined["zcta"].isna()]
matched_no_income = joined[joined["zcta"].notna() & joined["median_household_income"].isna()]

print(f"restaurants:                         {len(joined):,}")
print(f"  no zip code at all:                {int((~has_zip).sum()):,}")
print(f"  zip code that is NOT a Census ZCTA: {len(no_match):,} restaurants in {no_match.zipcode.nunique()} zip codes")
print(f"  matched a ZCTA but income is NA:    {len(matched_no_income):,} restaurants in {matched_no_income.zipcode.nunique()} ZCTAs")
print(f"  usable income:                     {int(joined.median_household_income.notna().sum()):,} "
      f"({joined.median_household_income.notna().mean():.1%})")
print(f"  in a top-coded ($250,000+) ZCTA:    {int(joined.income_topcoded.eq(True).sum()):,}")

print("\nZip codes with restaurants but no ZCTA (most restaurants first):")
print(no_match.groupby(["zipcode", "boro"]).size().sort_values(ascending=False).head(10).to_string())
print("\nZCTAs with restaurants but no income value:")
print(matched_no_income.groupby(["zipcode", "boro"]).size().sort_values(ascending=False).head(10).to_string())

joined.drop(columns="_merge").to_csv(f"{CLEAN}/restaurants_with_income.csv", index=False)

restaurants:                         27,463
  no zip code at all:                358
  zip code that is NOT a Census ZCTA: 146 restaurants in 20 zip codes
  matched a ZCTA but income is NA:    318 restaurants in 22 ZCTAs
  usable income:                     26,641 (97.0%)
  in a top-coded ($250,000+) ZCTA:    160

Zip codes with restaurants but no ZCTA (most restaurants first):
zipcode  boro     
10121    Manhattan    87
10281    Manhattan    18
10118    Manhattan     8
10105    Manhattan     5
10178    Manhattan     4
10041    Manhattan     4
10106    Manhattan     3
10055    Manhattan     2
10158    Manhattan     2
10123    Manhattan     2

ZCTAs with restaurants but no income value:
zipcode  boro     
11430    Queens       129
10020    Manhattan     46
10112    Manhattan     30
10119    Manhattan     29
11371    Queens        22
10169    Manhattan     18
10111    Manhattan      6
10167    Manhattan      6
10153    Manhattan      4
10103    Manhattan      4


## The surprise

I expected that a zip code either matches the Census or doesn't. Instead, the join showed **two different kinds of failure**, and both are concentrated in non-residential places:

- **146 restaurants in 20 zip codes have a zip that is not a Census ZCTA at all.** All of the top 10 are in Manhattan, led by 10121 (87 restaurants). These are zip codes the Postal Service assigns to single large buildings or complexes, and the Census does not draw ZCTAs for many of them. So "zip code" and "ZCTA" are not the same thing, as the P2 data dictionary warned, and here is the proof.
- **318 restaurants in 22 ZCTAs matched, but the income is missing** (the -666666666 "not available" code). The biggest is **11430 (129 restaurants), which is JFK Airport**, and 11371 (22) is LaGuardia. 10020 (46) is Rockefeller Center. These places have plenty of restaurants but almost no households, so there is no household income to report.

Together with 358 restaurants with no zip code, **822 restaurants (3.0%) have no usable income**, so 26,641 (97.0%) can be used in the analysis. The left join kept all 822 and counted them; an inner join would have dropped 504 of them without a message. These missing restaurants are not random: they are airport and office-tower restaurants, which matters when I compare low- and high-income areas later, because they will be excluded from both groups.

A smaller surprise: the 2023 Census table has 33,772 ZCTAs and 3,154 "not available" codes, while the 2022 table I used in P2 had 33,774 and 3,153. The same table changes slightly from year to year, which is why the year belongs in the citation.

---
# Self-scored rubric

| Criterion | Wt | Self-score (1–4) | Evidence |
|---|---|---|---|
| Acquisition code | ×2 | 4 | 1.1–1.6: 2 s rate limit, honest User-Agent, cache in `data/raw/` (bad responses never cached), `$limit`/`$offset` pagination with a row-count check, timeouts and retries, column checks; two demonstrated catches in 1.6; rate limit read from headers in 1.3 |
| Ethics & legality in practice | ×2 | 4 | 2.1 plan-vs-code table, 2.2 four drift items with reasons, 2.3 guardrails |
| Join design | ×2 | 4 | 4.1 table written above the merge, `validate="m:1"`, 4.2 alternatives ruled out, 4.3 guards against both fan-out and wreck |
| Verification suite | ×3 | 4 | 7 checks in `verify()`, each commented with what it protects against; proven on a fan-out and a wreck |
| Reproducibility & readability | ×1 | 4 | Restart-and-run-all with no errors; README updated; raw data untouched |

---
## AI use note

I used Claude to write the acquisition, join and verification code and the draft text in this notebook, based on my A8 plan and P2 notebook. I ran the notebook in Colab myself, checked every output, and the numbers in the written answers (row counts, match rate, the surprise) come from my own run on Nov 5, 2026. Claude tested the code against simulated API responses because it could not reach the live APIs.